# AstroTutor — Allineamento DPO (QLoRA + TRL)

Allinea `Qwen2.5-3B-Instruct` con DPO sulle 623 triplette di `alignment_data.json` e produce
un GGUF pronto per Ollama.

Runtime: **GPU L4**. Esegui le celle in ordine, dalla 1 alla 9.

> Perche' Liger, perche' non la T4, e la diagnosi degli OOM: vedi `report/report_2026_07_25.md`.

## 1 — Mount di Drive

Monta Google Drive. Prima di eseguire, carica a mano `data/alignment_data.json` in
`MyDrive/astrotutor/data/` (il repo e' privato, non clonabile da Colab).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 — Setup

Installa le librerie e disinstalla `torchao`: la versione preinstallata su Colab (0.10) e'
troppo vecchia per PEFT e farebbe fallire il merge alla cella 7.

In [ ]:
!pip install -q -U trl peft bitsandbytes datasets accelerate transformers sentencepiece huggingface_hub liger-kernel

# Colab preinstalla torchao 0.10, ma peft pretende >0.16 e solleva ImportError appena
# tocchi PeftModel (si vede al merge, cella 7). Qui torchao non serve: via.
!pip uninstall -q -y torchao

print("Setup completato.")

## 3 — Config e controlli

Definisce percorsi e iperparametri, poi verifica che GPU, Drive e dataset siano a posto
prima di iniziare.

In [ ]:
import gc, glob, shutil, subprocess, sys
from pathlib import Path
import torch

# --- Path ---
DRIVE_ROOT = Path("/content/drive/MyDrive/astrotutor")    # durevole: sopravvive alla disconnessione
WORK_ROOT  = Path("/content/astrotutor_work")             # volatile: veloce, per gli intermedi grossi

DATA_FILE   = DRIVE_ROOT / "data" / "alignment_data.json" # caricato a mano: il repo e' privato
MODELS_DIR  = DRIVE_ROOT / "models"
DPO_OUT     = MODELS_DIR / "dpo_out"                      # checkpoint (per riprendere dopo un crash)
ADAPTER_DIR = MODELS_DIR / "astrotutor-dpo-adapter"       # ~120 MB, l'unica cosa insostituibile
GGUF_Q4     = MODELS_DIR / "astrotutor-3b-dpo-Q4_K_M.gguf"
MODELFILE   = MODELS_DIR / "Modelfile"

MERGED_DIR = WORK_ROOT / "merged"                         # ~6 GB, usa e getta
GGUF_F16   = WORK_ROOT / "astrotutor-3b-dpo-f16.gguf"     # ~6 GB, usa e getta
LLAMA_DIR  = WORK_ROOT / "llama.cpp"
# Gli intermedi stanno in /content e non su Drive perché il mount è FUSE: scriverci
# 12 GB richiede minuti, e si rigenerano dall'adapter quando serve.

MODELS_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"   # stesso modello di qwen2.5:3b su Ollama

# --- Knob ---
USE_LIGER   = True    # False solo su GPU >= 24 GB: senza Liger servono ~5-7 GB di picco in più
PRECOMPUTE  = False   # log-prob del riferimento calcolati una volta sola invece che a ogni
                      # step. TRL lo RIFIUTA insieme a Liger ("Liger DPO loss does not support
                      # precomputing reference log probabilities"). Non è una perdita: serviva
                      # a risparmiare memoria sulla T4, e Liger risolve quel problema alla
                      # radice. Il costo è un forward in più per step (il riferimento = base
                      # con adapter disattivati), non un tensore in più.
BATCH_SIZE  = 1
GRAD_ACCUM  = 16      # batch effettivo = BATCH_SIZE * GRAD_ACCUM
GRAD_CKPT   = True    # PyTorch elimina le attivazioni intermedie dalla VRAM salvando solo alcuni "punti di controllo". Durante il backward pass, ricalcola al volo le attivazioni necessarie.
EPOCHS      = 2
MAX_LENGTH  = 3584    # cap di troncamento. Il max reale misurato sulle 623 triplette è ~3.020,
                      # quindi non tronca nulla. Abbassarlo NON libera memoria: con batch=1
                      # non c'è padding, il costo dipende dalla lunghezza reale della sequenza.
OPTIM       = "paged_adamw_8bit"
SAVE_STEPS  = 10

# --- Controlli ---
assert torch.cuda.is_available(), "Nessuna GPU: Runtime > Cambia tipo di runtime > GPU (L4)"
name  = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} — {total:.1f} GB")
if "T4" in name:
    print("⚠️  T4 (Turing): niente bf16 nativo, sarà lentissimo. Cambia runtime.")

assert DRIVE_ROOT.parent.is_dir(), "Drive non montato: esegui la cella 1"
assert DATA_FILE.exists(), (
    f"Dataset non trovato: {DATA_FILE}\n"
    "Carica alignment_data.json su Drive in MyDrive/astrotutor/data/ (vedi cella 1)"
)

from trl.import_utils import is_liger_kernel_available
print(f"liger-kernel disponibile: {is_liger_kernel_available()}")
if USE_LIGER and not is_liger_kernel_available():
    raise RuntimeError("liger-kernel non installato: rilancia la cella 2 (e riavvia il runtime)")
if not USE_LIGER and total < 24:
    print(f"⚠️  Liger disattivo su {total:.0f} GB: rischio OOM concreto.")
if USE_LIGER and PRECOMPUTE:
    PRECOMPUTE = False
    print("⚠️  Liger e precompute_ref_log_probs sono incompatibili: disattivo PRECOMPUTE.")

print(f"Output durevoli su: {MODELS_DIR}")

## 4 — Dataset e modello

Tre celle: carica e divide il dataset (95/5) · definisce il modello base in 4-bit, la
configurazione LoRA e il monitor di VRAM · costruisce il `DPOTrainer`.

In [ ]:
from datasets import load_dataset

# load_dataset("json", ...) gestisce sia array JSON che JSONL (il file è JSONL)
raw = load_dataset("json", data_files=str(DATA_FILE), split="train")
print(f"Triplette totali: {len(raw)}")
print("Strategie:", sorted(set(raw["strategy"])))

split    = raw.select_columns(["prompt", "chosen", "rejected"]).train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)} — Eval: {len(eval_ds)}")

# Sanity check sul formato conversazionale atteso da DPOTrainer
ex = train_ds[0]
assert isinstance(ex["prompt"], list) and ex["prompt"][0]["role"] == "system"
assert ex["chosen"][0]["role"] == "assistant"
print("Esempio domanda:", ex["prompt"][-1]["content"][:120])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback
from peft import LoraConfig

def load_base_model():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4", # NormalFloat4 (NF4)
        bnb_4bit_compute_dtype=torch.bfloat16,   # L4 (Ada) e A100 (Ampere): bf16 nativo
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        dtype=torch.bfloat16,
        device_map={"": 0},   # pin sulla prima GPU: con "auto" accelerate può spezzare il
                              # modello, e paged_adamw_8bit ha bug di sync su tensori sparsi
                              # tra device (l'"illegal memory access" in optimizer.step())
    )
    model.config.use_cache = False
    return model, AutoTokenizer.from_pretrained(BASE_MODEL)

LORA_CONFIG = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

print("Pronto.")

In [ ]:
from trl import DPOConfig, DPOTrainer

def make_trainer(smoke_steps=0):
    '''smoke_steps > 0 => run di prova: pochi step, niente eval, niente salvataggi,
    output in una cartella separata per non inquinare i checkpoint della run vera.'''
    kwargs = dict(
        output_dir=str(WORK_ROOT / "dpo_smoke") if smoke_steps else str(DPO_OUT),
        beta=0.1,
        learning_rate=5e-6,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_length=MAX_LENGTH,
        gradient_checkpointing=GRAD_CKPT,
        bf16=True,    # bf16 non ha bisogno di loss scaling: niente GradScaler da rompere
        fp16=False,
        optim=OPTIM,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        report_to="none",
        use_liger_kernel=USE_LIGER,
    )
    if smoke_steps:
        kwargs.update(max_steps=smoke_steps, logging_steps=1,
                      save_strategy="no", eval_strategy="no")
    else:
        kwargs.update(logging_steps=5,
                      save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=3,
                      eval_strategy="epoch", per_device_eval_batch_size=1)
        # per_device_eval_batch_size: il default (8) causava OOM in eval — stesso tensore
        # dei logits del training ma batch 8x (18.22 GiB richiesti il 21/07)
    if PRECOMPUTE:
        kwargs.update(precompute_ref_log_probs=True, precompute_ref_batch_size=1)

    model, tokenizer = load_base_model()
    trainer = DPOTrainer(
        model=model,
        args=DPOConfig(**kwargs),
        train_dataset=train_ds,
        eval_dataset=None if smoke_steps else eval_ds,
        processing_class=tokenizer,
        peft_config=LORA_CONFIG,
    )
    return trainer, tokenizer

def free():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

print("Pronto.")

## 5 — Smoke test (3 step)

Run di prova da 2 minuti. Serve a vedere se il picco di VRAM sta nel budget **prima** di
lanciare l'ora di training vera.

In [ ]:
free()
trainer, _ = make_trainer(smoke_steps=3)
trainer.train()

print(f"\n✓ Smoke test finito. Picco VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print("  Controlla sopra se nei log compare `rewards/accuracies`.")

del trainer
free()
shutil.rmtree(WORK_ROOT / "dpo_smoke", ignore_errors=True)

## 6 — Training

2 epoche, ~1 ora su L4. A fine corsa salva l'adapter LoRA su Drive. Se il runtime si
disconnette, rilancia questa cella: riprende dall'ultimo checkpoint.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

free()
trainer, tokenizer = make_trainer()

last_checkpoint = get_last_checkpoint(str(DPO_OUT)) if DPO_OUT.is_dir() else None
if last_checkpoint:
    print(f"Riprendo dal checkpoint: {last_checkpoint}")

trainer.train(resume_from_checkpoint=last_checkpoint)

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"\n✓ Adapter LoRA salvato su Drive: {ADAPTER_DIR}")

del trainer
free()

## 7 — Merge dell'adapter nel modello base

Fonde i pesi LoRA nel modello base, su GPU se c'e' VRAM libera.

In [ ]:
from peft import PeftModel

free()
freemem = torch.cuda.mem_get_info()[0] / 1e9
device  = "cuda" if freemem >= 10 else "cpu"    # il 3B in bf16 pesa ~6,2 GB
print(f"Merge su {device} (VRAM libera: {freemem:.1f} GB)")

base   = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, device_map=device)
merged = PeftModel.from_pretrained(base, str(ADAPTER_DIR)).merge_and_unload()
merged.save_pretrained(str(MERGED_DIR))
AutoTokenizer.from_pretrained(str(ADAPTER_DIR)).save_pretrained(str(MERGED_DIR))
print(f"✓ Merge completato: {MERGED_DIR}")

del base, merged
free()

## 8 — Conversione in GGUF Q4_K_M

Quattro celle: clona llama.cpp · patcha un bug del tokenizer di transformers · converte in
GGUF f16 e compila `llama-quantize` · quantizza in Q4_K_M su Drive.

In [ ]:
if LLAMA_DIR.exists():
    shutil.rmtree(LLAMA_DIR)
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_DIR}
!pip install -q -r {LLAMA_DIR}/requirements/requirements-convert_hf_to_gguf.txt

In [ ]:
def patch_transformers_tokenizer():
    '''Patch difensiva a un bug in _set_model_specific_special_tokens (extra_special_tokens
    è una list invece di un dict) che rompe convert_hf_to_gguf.py al caricamento del tokenizer.
    Due punti di rottura noti (.keys() e .items()): patchiamo il sorgente installato invece
    di inseguire versione per versione.'''
    import transformers
    tub = Path(transformers.__file__).parent / "tokenization_utils_base.py"
    src = tub.read_text(encoding="utf-8")
    patches = [
        ("self.SPECIAL_TOKENS_ATTRIBUTES = self.SPECIAL_TOKENS_ATTRIBUTES + list(special_tokens.keys())",
         "self.SPECIAL_TOKENS_ATTRIBUTES = self.SPECIAL_TOKENS_ATTRIBUTES + (list(special_tokens.keys()) if isinstance(special_tokens, dict) else [])"),
        ("for key, value in special_tokens.items():",
         "for key, value in (special_tokens.items() if isinstance(special_tokens, dict) else []):"),
    ]
    changed = False
    for old, new in patches:
        if old in src:
            src = src.replace(old, new); changed = True
            print(f"Patchata: {old[:60]}")
        elif new in src:
            print(f"Già patchata: {old[:60]}")
        else:
            print(f"⚠️  Pattern non trovato: {old[:60]}")
    if changed:
        tub.write_text(src, encoding="utf-8")

patch_transformers_tokenizer()

In [ ]:
# HF -> GGUF f16
!python {LLAMA_DIR}/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_F16} --outtype f16

# Build del solo target llama-quantize
!cmake -S {LLAMA_DIR} -B {LLAMA_DIR}/build -DLLAMA_CURL=OFF
!cmake --build {LLAMA_DIR}/build --target llama-quantize -j

In [ ]:
matches = [m for m in glob.glob(str(LLAMA_DIR / "build" / "**" / "llama-quantize*"), recursive=True)
           if not m.endswith((".o", ".obj"))]
assert matches, "llama-quantize non trovato dopo il build"

subprocess.run([matches[0], str(GGUF_F16), str(GGUF_Q4), "Q4_K_M"], check=True)
print(f"\n✓ GGUF su Drive: {GGUF_Q4} ({GGUF_Q4.stat().st_size/1e6:.0f} MB)")

## 9 — Modelfile per Ollama

Scrive il Modelfile accanto al GGUF e stampa i comandi per installare il modello in locale.

In [ ]:
modelfile = """FROM ./astrotutor-3b-dpo-Q4_K_M.gguf

TEMPLATE \"\"\"{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
\"\"\"
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
"""
MODELFILE.write_text(modelfile, encoding="utf-8")
print(f"✓ Modelfile scritto: {MODELFILE}")
print()
print("Ora, in locale:")
print("  1. scarica da Drive (MyDrive/astrotutor/models/): astrotutor-3b-dpo-Q4_K_M.gguf + Modelfile")
print("  2. mettili in progetto_IR/models/")
print("  3. cd models && ollama create astrotutor-dpo -f Modelfile")